In [ ]:
import numpy as np
import torch as th
import matplotlib.pyplot as plt
from cnn_surgery.utils.load_dataset import load_dataset
from cnn_surgery.lenses.regressor_lens import get_regressor_lens

In [ ]:
early = load_dataset('mnist', metrics_file='metrics_merged_early.csv', load_class_acc=True, stage='early')
middle = load_dataset('mnist', metrics_file='metrics_merged_middle.csv', load_class_acc=True, stage='middle')
final = load_dataset('mnist', metrics_file='metrics_merged_final.csv', load_class_acc=True, stage='final')

train_early, test_early, val_early = early
train_middle, test_middle, val_middle = middle
train_final, test_final, val_final = final

weights_train = np.concatenate([train_early[0], train_middle[0], train_final[0]])
weights_val = np.concatenate([val_early[0], val_middle[0], val_final[0]])

accuracies_train = np.concatenate([train_early[1][:, -10:], train_middle[1][:, -10:], train_final[1][:, -10:]])
accuracies_val = np.concatenate([val_early[1][:, -10:], val_middle[1][:, -10:], val_final[1][:, -10:]])

In [ ]:
meta_network = get_regressor_lens(weights_train, accuracies_train, weights_val, accuracies_val, device='cpu')

In [ ]:
# pick a random network to play around with
MODEL_IDX = -516
model_weights = weights_val[MODEL_IDX]
model_weights_tensor = th.tensor(model_weights, dtype=th.float32, requires_grad=True).unsqueeze(0)
model_weights_tensor.retain_grad()
model_accuracies = accuracies_val[MODEL_IDX]
pred = meta_network(model_weights_tensor).squeeze(0)

In [ ]:
# plot
fig, ax = plt.subplots(figsize=(10, 6))

# Define the positions for the bars
indices = np.arange(len(pred))
width = 0.4  # Width of the bars

# Plot predicted accuracies
ax.bar(indices - width/2, pred.detach().numpy(), width, label='Predicted Accuracies')

# Plot actual accuracies
ax.bar(indices + width/2, model_accuracies, width, label='Actual Accuracies')

# Add labels, title, and legend
ax.set_title("Predicted vs Actual Accuracies")
ax.set_xlabel("Class Index")
ax.set_ylabel("Accuracy")
ax.set_xticks(indices)
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
def simple_loss_fn(pred, target_class):
    """ A simple loss that encourages unlearning of a specific class."""
    return pred[target_class]

def boost_loss_fn(pred, target_class, beta=0.2):
    """ Encourages unlearning of a specific class, whilst boosting accuracy on other classes."""
    mask = -th.ones_like(pred) * beta
    mask[target_class] = 1
    return (mask * pred).sum()

def unlearning_and_faithfulness_loss(pred, true, target_class, alpha=1.0):
    """ Encourages unlearning of a specific class, whilst maintaining faithfulness to the true accuracies."""
    unlearning_loss = simple_loss_fn(pred, target_class)
    faithfulness_loss = ((true - pred)**2).mean()
    return unlearning_loss + alpha * faithfulness_loss

In [ ]:
loss_fn = boost_loss_fn
directions = []
true = th.tensor(model_accuracies)
for target_class in range(10):
    model_weights_tensor.grad = None
    pred = meta_network(model_weights_tensor).squeeze(0)
    loss = loss_fn(pred, target_class=target_class)
    loss.backward()
    directions.append(-model_weights_tensor.grad.squeeze(0).detach().numpy())
    meta_network.zero_grad()

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Compute cosine similarity matrix
cos_sim_matrix = cosine_similarity(np.array(directions))

# Plot the cosine similarity matrix
fig, ax = plt.subplots(figsize=(8, 6))
cax = ax.matshow(cos_sim_matrix, cmap='viridis', vmin=-1, vmax=1)

# Add colorbar
fig.colorbar(cax)

# Add labels and title
ax.set_title("Cosine Similarity Matrix of gradient directions\nBoost loss (beta=1.0)")
ax.set_xlabel("Target Class")
ax.set_ylabel("Target Class")
ax.set_xticks(range(len(directions)))
ax.set_yticks(range(len(directions)))

plt.show()

In [ ]:
from sklearn.decomposition import PCA

# Normalize directions
# Normalize directions
directions = [direction / np.linalg.norm(direction) for direction in directions]

# Perform PCA on the normalized directions
pca = PCA(n_components=2)
directions_pca = pca.fit_transform(np.array(directions))

# Plot the 2D PCA
fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(directions_pca[:, 0], directions_pca[:, 1], c=range(len(directions_pca)), cmap='tab10', s=100)

# Add labels and title
ax.set_title("2D PCA of Gradient Directions")
ax.set_xlabel("Principal Component 1")
ax.set_ylabel("Principal Component 2")
ax.legend(handles=scatter.legend_elements()[0], labels=[f"Class {i}" for i in range(len(directions_pca))])

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# directions_pca: (N, 2)
N = directions_pca.shape[0]

# Use a larger palette than tab10 if needed
cmap = plt.cm.get_cmap("tab20", max(3, min(20, N)))
colors = cmap(np.arange(N))

# Make layout robust
fig, ax = plt.subplots(figsize=(7, 7), constrained_layout=True)

# Quiver from the origin
O = np.zeros(N)
ax.quiver(
    O, O,
    directions_pca[:, 0], directions_pca[:, 1],
    angles="xy", scale_units="xy", scale=1,
    color=colors, width=0.004, headwidth=3.5, headlength=5, zorder=2
)

# Optional: mark arrow tips for visibility
ax.scatter(directions_pca[:, 0], directions_pca[:, 1], s=15, color=colors, zorder=3)

# Labels at tips (don't let them expand autoscale)
for i, (x, y) in enumerate(directions_pca):
    ax.text(x, y, f"Class {i}", fontsize=9, color=colors[i],
            ha="left", va="bottom", clip_on=False)

# Axes lines for reference
ax.axhline(0, color="0.85", lw=0.8, zorder=0)
ax.axvline(0, color="0.85", lw=0.8, zorder=0)

# Keep equal scaling and add padding so nothing hugs the frame
m = float(np.max(np.abs(directions_pca))) if N else 1.0
pad = 0.15 * m if m > 0 else 1.0
ax.set_xlim(-m - pad, m + pad)
ax.set_ylim(-m - pad, m + pad)
ax.set_aspect("equal", adjustable="box")

# Titles/labels
ax.set_title("PCA of Gradient Directions")
ax.set_xlabel("Principal Component 1")
ax.set_ylabel("Principal Component 2")

plt.show()

# Tuning weighting
As we can see the variation in directions seem to be quitte dependent on the value of a weighting parameter. Alpha for faithful loss and beta for boost loss. Let's explore how we might tune those.

As with random directions, perhaps we would like the directions to be as orthogonal as possible.Thus perhaps a good indicator for tuning is the mean off-diagonal for the cosine similarity direction matrix we made above.

In [ ]:
def get_direction(loss_fn, target_class, meta_network, model_weights_tensor, parameter):
    """Compute the gradient direction for a specific target class."""
    model_weights_tensor.grad = None
    pred = meta_network(model_weights_tensor).squeeze(0)
    loss = loss_fn(pred, target_class, parameter)
    loss.backward()
    direction = -model_weights_tensor.grad.squeeze(0).detach().numpy()
    meta_network.zero_grad()
    return direction

def get_directions(loss_fn, meta_network, model_weights_tensor, num_classes=10, parameter=1.0):
    """Compute gradient directions for all target classes."""
    directions = []
    for target_class in range(num_classes):
        direction = get_direction(loss_fn, target_class, meta_network, model_weights_tensor, parameter)
        directions.append(direction)
    return np.array(directions)

In [ ]:
def compute_mean_off_diag(loss_fn, meta_network, model_weights_tensor, parameter=0.1):
    """
    Compute the mean of the absolute values of the off-diagonal elements 
    of the cosine similarity matrix of gradient directions.

    Args:
        loss_fn: The loss function to use.
        meta_network: The meta network model.
        model_weights_tensor: The tensor of model weights.
        parameter: The parameter for the loss function (default is 0.1).

    Returns:
        float: The mean of the absolute values of the off-diagonal elements.
    """
    directions = get_directions(loss_fn, meta_network, model_weights_tensor, parameter=parameter)
    similarity = cosine_similarity(directions)
    lower_tri = np.tril(similarity, k=-1)
    mean_off_diag = abs(lower_tri[lower_tri != 0]).mean()
    return mean_off_diag

parameters = np.arange(-1, 2, 0.01)
mean_off_diags = [compute_mean_off_diag(boost_loss_fn, meta_network, model_weights_tensor, parameter=param) for param in parameters]

# Plot the results
plt.figure(figsize=(10, 6))
plt.plot(parameters, mean_off_diags, label="Mean Off-Diagonal")
plt.axvline(x=0.1, color='red', linestyle='--', label='x=0.1')
plt.axvline(x=0.25, color='red', linestyle='--', label='x=0.25')
plt.xlabel("Parameter")
plt.ylabel("Mean Off-Diagonal")
plt.title("Mean Off-Diagonal vs Parameter")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

The exact shape of the dip is depended on the specific instance of the meta-network. But the dip seems to be consistently around the red lines. I would recommend a range for beta between 0.1 and 0.25, with perhaps a slight edge at ~0.2.

Let's repeat the experiment for an aggregate of 1000 models from the validation set

In [ ]:
# Function to compute mean off-diagonal for all models in the validation set
def compute_mean_off_diag_for_all_models(loss_fn, meta_network, weights_val, accuracies_val, parameters):
    results = []
    for model_idx in range(1000):
        model_weights_tensor = th.tensor(weights_val[-model_idx], dtype=th.float32, requires_grad=True).unsqueeze(0)
        model_weights_tensor.retain_grad()
        true = th.tensor(accuracies_val[model_idx])
        mean_off_diags = [
            compute_mean_off_diag(loss_fn, meta_network, model_weights_tensor, parameter=param)
            for param in parameters
        ]
        results.append(mean_off_diags)
    return np.array(results)

# Compute mean off-diagonal for the entire validation set
all_mean_off_diags = compute_mean_off_diag_for_all_models(
    boost_loss_fn, meta_network, weights_val, accuracies_val, parameters
)

In [ ]:
# Plot the results for the entire validation set
plt.figure(figsize=(10, 6))
for model_idx, mean_off_diags in enumerate(all_mean_off_diags):
    plt.plot(parameters, mean_off_diags, label=f"Model {model_idx}")
plt.xlabel("Parameter")
plt.ylabel("Mean Off-Diagonal")
plt.title("Mean Off-Diagonal vs Parameter (Validation Set)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Compute mean and standard deviation across all models
mean_off_diags_aggregate = all_mean_off_diags.mean(axis=0)
std_off_diags_aggregate = all_mean_off_diags.std(axis=0)

# Plot the mean with confidence bounds
plt.figure(figsize=(10, 6))
plt.plot(parameters, mean_off_diags_aggregate, label="Mean Off-Diagonal", color="blue")
plt.fill_between(parameters, 
                 mean_off_diags_aggregate - std_off_diags_aggregate, 
                 mean_off_diags_aggregate + std_off_diags_aggregate, 
                 color="blue", alpha=0.2, label="Confidence Bound (±1 STD)")

# Add labels, title, and legend
plt.xlabel("Parameter")
plt.ylabel("Mean Off-Diagonal")
plt.title("Aggregate Mean Off-Diagonal with Confidence Bounds for 1000 models")
plt.axvline(x=0.1, color='red', linestyle='--', label='x=0.1')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
def unlearn(step_size=0.001, n_steps=100):
    model_weights_tensor.grad = None
    pred = meta_network(model_weights_tensor).squeeze(0)
    loss = loss_fn(pred, true, target_class=target_class)
    loss.backward()
    directions.append(-model_weights_tensor.grad.squeeze(0).detach().numpy())
    meta_network.zero_grad()